# FLAIR Quick Start

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TakatoHonda/FLAIR/blob/main/examples/quickstart.ipynb)

**Factored Level And Interleaved Ridge** — zero-hyperparameter time series forecasting.

This notebook covers:
1. Installation
2. Point forecast and prediction intervals
3. Visualization
4. Different frequencies
5. Reproducibility with `seed`
6. Class API for batch forecasting
7. Working with pandas
8. Real data: AirPassengers

## 0. Install

In [ ]:
!pip install -q "flaircast>=0.2.0"
import flaircast; print(f"flaircast {flaircast.__version__}")
assert flaircast.__version__ >= "0.2.0", "Please restart runtime: Runtime > Restart session"

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from flaircast import forecast, FLAIR

## 1. Generate sample data

An hourly time series with daily seasonality (period=24), trend, and noise.

In [ ]:
rng = np.random.RandomState(0)
n = 500
t = np.arange(n, dtype=float)

y = 100 + 0.05 * t + 20 * np.sin(2 * np.pi * t / 24) + rng.normal(0, 3, n)

plt.figure(figsize=(12, 3))
plt.plot(y, linewidth=0.8)
plt.title("Hourly time series (500 observations)")
plt.xlabel("Hour")
plt.ylabel("Value")
plt.tight_layout()
plt.show()

## 2. Forecast with prediction intervals

`forecast()` returns `(n_samples, horizon)` sample paths. Compute any summary you need: mean, median, quantiles.

In [ ]:
horizon = 48  # 2 days ahead
samples = forecast(y, horizon=horizon, freq="H", seed=42)

print(f"samples.shape = {samples.shape}")  # (200, 48)

point = samples.mean(axis=0)
lo_10, hi_90 = np.percentile(samples, [10, 90], axis=0)
lo_25, hi_75 = np.percentile(samples, [25, 75], axis=0)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))

# History (last 5 days)
history_len = 120
ax.plot(range(history_len), y[-history_len:],
        color="black", linewidth=0.8, label="History")

# Forecast
fc_x = range(history_len, history_len + horizon)
ax.plot(fc_x, point, color="#2563eb", linewidth=1.5, label="Point forecast")
ax.fill_between(fc_x, lo_25, hi_75, alpha=0.3, color="#2563eb", label="50% interval")
ax.fill_between(fc_x, lo_10, hi_90, alpha=0.15, color="#2563eb", label="80% interval")

ax.axvline(x=history_len, color="gray", linestyle="--", linewidth=0.8)
ax.legend(loc="upper left")
ax.set_title("FLAIR forecast — hourly data, 48-step horizon")
ax.set_xlabel("Hour")
ax.set_ylabel("Value")
plt.tight_layout()
plt.show()

## 3. Different frequencies

FLAIR auto-selects the period from the frequency string. Just change `freq`.

In [ ]:
# Monthly data with annual seasonality
rng = np.random.RandomState(1)
t_m = np.arange(120, dtype=float)  # 10 years
y_monthly = 1000 + 200 * np.sin(2 * np.pi * t_m / 12) + 2 * t_m + rng.normal(0, 30, 120)

samples_m = forecast(y_monthly, horizon=24, freq="M", seed=42)
point_m = samples_m.mean(axis=0)
lo_m, hi_m = np.percentile(samples_m, [10, 90], axis=0)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(range(120), y_monthly, color="black", linewidth=0.8, label="History")
fc_x_m = range(120, 144)
ax.plot(fc_x_m, point_m, color="#dc2626", linewidth=1.5, label="Point forecast")
ax.fill_between(fc_x_m, lo_m, hi_m, alpha=0.2, color="#dc2626", label="80% interval")
ax.axvline(x=120, color="gray", linestyle="--", linewidth=0.8)
ax.legend(loc="upper left")
ax.set_title("Monthly data — 24-month horizon")
ax.set_xlabel("Month")
ax.set_ylabel("Value")
plt.tight_layout()
plt.show()

## 4. Reproducibility

Pass `seed` for identical results across runs.

In [ ]:
a = forecast(y, horizon=24, freq="H", seed=42)
b = forecast(y, horizon=24, freq="H", seed=42)

print(f"Same seed, identical output: {np.array_equal(a, b)}")  # True

c = forecast(y, horizon=24, freq="H", seed=99)
print(f"Different seeds differ:      {not np.array_equal(a, c)}")  # True

## 5. Class API for batch forecasting

`FLAIR` stores `freq`, `n_samples`, and `seed` so you don't repeat them in a loop.

In [ ]:
model = FLAIR(freq="D", n_samples=100, seed=42)

# 5 daily time series
rng = np.random.RandomState(0)
series_list = [
    50 + 10 * np.sin(2 * np.pi * np.arange(200) / 7) + rng.normal(0, 2, 200)
    for _ in range(5)
]

fig, axes = plt.subplots(1, 5, figsize=(16, 3), sharey=True)
for i, (s, ax) in enumerate(zip(series_list, axes)):
    samp = model.predict(s, horizon=14)
    pt = samp.mean(axis=0)
    lo, hi = np.percentile(samp, [10, 90], axis=0)

    ax.plot(range(50), s[-50:], color="black", linewidth=0.7)
    fc_x = range(50, 64)
    ax.plot(fc_x, pt, color="#2563eb", linewidth=1.2)
    ax.fill_between(fc_x, lo, hi, alpha=0.2, color="#2563eb")
    ax.axvline(x=50, color="gray", linestyle="--", linewidth=0.5)
    ax.set_title(f"Series {i+1}", fontsize=10)

fig.suptitle("Batch forecasting with FLAIR class", fontsize=12)
plt.tight_layout()
plt.show()

## 6. Working with pandas

Pass `.values` from a pandas Series.

In [ ]:
import pandas as pd

dates = pd.date_range("2024-01-01", periods=365, freq="D")
rng = np.random.RandomState(2)
values = 80 + 15 * np.sin(2 * np.pi * np.arange(365) / 7) + rng.normal(0, 3, 365)
ts = pd.Series(values, index=dates, name="daily_sales")

samples_pd = forecast(ts.values, horizon=14, freq="D", seed=42)
point_pd = samples_pd.mean(axis=0)
lo_pd, hi_pd = np.percentile(samples_pd, [10, 90], axis=0)

fc_dates = pd.date_range(ts.index[-1] + pd.Timedelta(days=1), periods=14, freq="D")

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(ts.index[-60:], ts.values[-60:], color="black", linewidth=0.8, label="History")
ax.plot(fc_dates, point_pd, color="#2563eb", linewidth=1.5, label="Forecast")
ax.fill_between(fc_dates, lo_pd, hi_pd, alpha=0.2, color="#2563eb", label="80% interval")
ax.axvline(x=ts.index[-1], color="gray", linestyle="--", linewidth=0.8)
ax.legend()
ax.set_title("FLAIR with pandas")
plt.tight_layout()
plt.show()

---

**Links**: [GitHub](https://github.com/TakatoHonda/FLAIR) | [PyPI](https://pypi.org/project/flaircast/) | [API Reference](https://github.com/TakatoHonda/FLAIR#api-reference)

## 7. Real data: AirPassengers

The classic Box-Jenkins airline passengers dataset (144 monthly observations, 1949-1960). Trend + multiplicative seasonality — a good fit for FLAIR.

In [ ]:
import pandas as pd

# Load AirPassengers from GitHub (no extra dependencies needed)
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv"
df = pd.read_csv(url, parse_dates=["Month"], index_col="Month")
y_air = df["Passengers"].values.astype(float)

print(f"AirPassengers: {len(y_air)} monthly observations")
print(f"Range: {y_air.min():.0f} - {y_air.max():.0f}")

# Hold out last 12 months for evaluation
train, actual = y_air[:-12], y_air[-12:]
horizon = 12

samples_air = forecast(train, horizon=horizon, freq="M", seed=42)
point_air = samples_air.mean(axis=0)
lo_air, hi_air = np.percentile(samples_air, [10, 90], axis=0)

# Plot
fig, ax = plt.subplots(figsize=(12, 4))
n_train = len(train)
ax.plot(range(n_train), train, color="black", linewidth=0.8, label="Train")
fc_x = range(n_train, n_train + horizon)
ax.plot(fc_x, actual, color="gray", linewidth=1.2, linestyle="--", label="Actual")
ax.plot(fc_x, point_air, color="#2563eb", linewidth=1.5, label="FLAIR forecast")
ax.fill_between(fc_x, lo_air, hi_air, alpha=0.2, color="#2563eb", label="80% interval")
ax.axvline(x=n_train, color="gray", linestyle="--", linewidth=0.5)
ax.legend(loc="upper left")
ax.set_title("AirPassengers — 12-month ahead forecast")
ax.set_xlabel("Month index")
ax.set_ylabel("Passengers")
plt.tight_layout()
plt.show()

# MASE (vs seasonal naive)
naive_errors = np.abs(train[12:] - train[:-12])
scale = naive_errors.mean()
mase = np.mean(np.abs(actual - point_air)) / scale
print(f"\nMASE: {mase:.3f} (< 1.0 means better than Seasonal Naive)")